In [ ]:
!pip install openai numpy tensorflow matplotlib logging

In [ ]:
# Cell 1: Imports & Setup
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import logging
from openai import OpenAI

# Configure logging
logging.basicConfig(filename="results.log",
                    level=logging.INFO,
                    format="%(asctime)s - %(message)s")

# Initialize OpenAI client
client = OpenAI()

In [ ]:
# Cell 2: Load and preprocess CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

class_names = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]

In [ ]:
# Cell 3: Build CNN model
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

In [ ]:
# Cell 4: Train model
history = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

In [ ]:
# Cell 5: Evaluate model
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# Cell 6: Visualize training history
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Cell 7: Predict a sample and visualize
sample_idx = 0
sample = np.expand_dims(x_test[sample_idx], axis=0)
prediction = model.predict(sample)
predicted_class = class_names[prediction.argmax()]

plt.imshow(x_test[sample_idx])
plt.title(f"Predicted: {predicted_class}")
plt.axis("off")
plt.show()

print("Predicted class:", predicted_class)

In [ ]:
# Cell 8: Explain prediction with OpenAI
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"The CNN predicted '{predicted_class}' for an image. Explain why CNNs are good at image classification in simple terms."}
    ]
)

explanation = response.choices[0].message.content
print("OpenAI Explanation:", explanation)

# Log results
logging.info(f"Prediction: {predicted_class}, Explanation: {explanation}")